## CSV and Excel Files - Structured Data

In [ ]:
import pandas as pd
import os

In [2]:
os.makedirs("data/structured_files", exist_ok = True)

In [9]:
data = {
    "Product": ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'WebCam'],
    "Category": ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    "Price": [1200, 25, 45, 300, 80],
    "Stock": [50, 200, 150, 75, 100],
    "Description": [
        'A high-performance laptop with 16GB RAM and 512GB SSD.',
        'A wireless mouse with ergonomic design.',
        'A mechanical keyboard with RGB backlighting.',
        'A 27-inch 4K monitor with HDR support.',
        'A high-definition webcam for video conferencing.'
    ]
}

# Save as CSV
df = pd.DataFrame(data)
df.to_csv("data/structured_files/products.csv", index = False)

In [10]:
with pd.ExcelWriter("data/structured_files/inventory.xlsx") as writer:
    df.to_excel(writer, sheet_name = "Products", index = False)

    summary_data = {
        "Category": ["Electronics", "Accessories"],
        "Total_Items": [3, 2],
        "Total_Value": [1389.97, 70.00]
    }

    pd.DataFrame(summary_data).to_excel(writer, sheet_name = "Summary", index = False) 

## CSV Processing

In [6]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

c:\Users\CHITTA\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Method 1: Using CSVLoader - Each Row Becomes a document
print("CSVLoader - Row-based Documents")
csv_loader = CSVLoader(
    file_path = "data/structured_files/products.csv", 
    encoding = "utf-8", 
    csv_args = {
        "delimiter": ",", 
        "quotechar": '"'
    }
)

csv_docs = csv_loader.load()
print(f"Number of documents loaded: {len(csv_docs)}")
print("\nFirst document content:")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")
print("\nSecond document content:")
print(f"Content: {csv_docs[1].page_content}")
print(f"Metadata: {csv_docs[1].metadata}")

CSVLoader - Row-based Documents
Number of documents loaded: 5

First document content:
Content: Product: Laptop
Category: Electronics
Price: 1200
Stock: 50
Description: A high-performance laptop with 16GB RAM and 512GB SSD.
Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}

Second document content:
Content: Product: Mouse
Category: Accessories
Price: 25
Stock: 200
Description: A wireless mouse with ergonomic design.
Metadata: {'source': 'data/structured_files/products.csv', 'row': 1}


In [12]:
# Method 2: Custom CSV processing for better control
from typing import List
from langchain_core.documents import Document

print("\nCustom CSV processing")
def process_csv_intelligently(filepath: str) -> List[Document]:
    """Process CSV with intelligent document creation."""
    df = pd.read_csv(filepath)
    documents = []
    
    for i, row in df.iterrows():
        content = f""" Product Information:
        Name: {row['Product']}
        Category: {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']}
        Description: {row['Description']}"""
        metadata = {"source": filepath, "row_idx": i, "product": row['Product'], "category": row['Category'], "price": row['Price'], "stock": row['Stock'], "data_type": "product_info"}
        documents.append(Document(page_content=content, metadata=metadata))
    
    return documents


Custom CSV processing


In [13]:
process_csv_intelligently("data/structured_files/products.csv")

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_idx': 0, 'product': 'Laptop', 'category': 'Electronics', 'price': 1200, 'stock': 50, 'data_type': 'product_info'}, page_content=' Product Information:\n        Name: Laptop\n        Category: Electronics\n        Price: $1200\n        Stock: 50\n        Description: A high-performance laptop with 16GB RAM and 512GB SSD.'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_idx': 1, 'product': 'Mouse', 'category': 'Accessories', 'price': 25, 'stock': 200, 'data_type': 'product_info'}, page_content=' Product Information:\n        Name: Mouse\n        Category: Accessories\n        Price: $25\n        Stock: 200\n        Description: A wireless mouse with ergonomic design.'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_idx': 2, 'product': 'Keyboard', 'category': 'Accessories', 'price': 45, 'stock': 150, 'data_type': 'product_info'}, page_content=' Product Information

## CSV Processing Strategies:
| Row-Based(CSV Loader) | Intelligent Processing |
| --- | --- |
| ✅ Simple one-row-one-document | ✅ Preserves relationships |
| ✅ Good for record lookups | ✅ Creates summaries |
| ❌ Loses table context | ✅ Rich metadata |
|| ✅ Better for Q&A |

## Excel Processing

In [14]:
# Method 1: Using pandas for full control over Excel files
print("\nProcessing Excel with pandas")
def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Process Excel file with sheet awareness"""
    xls = pd.ExcelFile(filepath)
    documents = []
    
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name = sheet_name)

        content = f"Sheet: {sheet_name}"
        content += f"\nColumns: {', '.join(df.columns)}"
        content += f"\nNumber of Rows: {len(df)}\n\n"
        content += df.to_string(index = False)
        metadata = {
            "source": filepath, 
            "sheet_name": sheet_name, 
            "data_type": "excel_sheet",
            "num_rows": len(df),
            "num_columns": len(df.columns)
        }
        documents.append(Document(page_content = content, metadata = metadata))
    
    return documents 


Processing Excel with pandas


In [15]:
excel_docs = process_excel_with_pandas("data/structured_files/inventory.xlsx")
print(f"Number of documents loaded from Excel: {len(excel_docs)}")

Number of documents loaded from Excel: 2


In [16]:
excel_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'data_type': 'excel_sheet', 'num_rows': 5, 'num_columns': 5}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nNumber of Rows: 5\n\n Product    Category  Price  Stock                                            Description\n  Laptop Electronics   1200     50 A high-performance laptop with 16GB RAM and 512GB SSD.\n   Mouse Accessories     25    200                A wireless mouse with ergonomic design.\nKeyboard Accessories     45    150           A mechanical keyboard with RGB backlighting.\n Monitor Electronics    300     75                 A 27-inch 4K monitor with HDR support.\n  WebCam Electronics     80    100       A high-definition webcam for video conferencing.'),
 Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'data_type': 'excel_sheet', 'num_rows': 2, 'num_columns': 3}, page_content='Sheet: Summary\n

In [17]:
# Method 2: UnstructuredExcelLoader - Each Sheet Becomes a document
from langchain_community.document_loaders import UnstructuredExcelLoader


print("\nUnstructuredExcelLoader - Sheet-based Documents")
try:
    excel_loader = UnstructuredExcelLoader(
        file_path = "data/structured_files/inventory.xlsx",
        mode = "elements",
    )
    unstructured_excel_docs = excel_loader.load()
    print(f"Number of documents loaded from Excel: {len(unstructured_excel_docs)}")
    print("\nFirst document content:")
    print(f"Content: {unstructured_excel_docs[0].page_content}")
except Exception as e:
    print(f"Error loading Excel file: {e}")


UnstructuredExcelLoader - Sheet-based Documents
Error loading Excel file: No module named 'msoffcrypto'
